هذه نسخة يومية مستخرجة من دفتر سلطان المجمع المرفق. حُفظ الكود والمخرجات وأرقام التنفيذ كما وردت، ولم تُعد الخلايا للتشغيل أثناء التقسيم. الأصل الكامل محفوظ في [دفتر المشروع المجمع](../notebooks/Sultan_Training_Project.ipynb).

وصلت مخرجات محاولة جديدة للاب 06 بعد استعادة مساحة العمل: تسعة فحوص ناجحة، وسبعة صفوف معزولة، و75 صفًا معتمدًا. مخرجات التدفق وبقية الأيام محفوظة من التشغيل السابق؛ هذه ليست إعادة تشغيل كاملة للدورة. ملف ZIP للمحاولة الجديدة لم يصل بعد. راجع [تفاصيل الدليل](../reports/review/quality_rerun_notebook.json) و[كود الاستعادة والفحص](../scripts/colab_recovery/README.txt).

بيانات `metadata.masar` وتوقيتات `metadata.execution` القديمة موروثة من دفتر الدورة المرجعي؛ أزيلت من المقتطف حتى لا تُنسب إلى تشغيل سلطان. تبقى في الأصل غير المعدل لأغراض التتبع. راجع [سجل التقسيم](../notebooks/README.md).

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 4 · Streaming, quality and governance</h1><p>Receive Kafka events with a persistent checkpoint; reconcile delivery and event identities; validate, quarantine and recheck a batch; document quality and governance decisions.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 4 · التدفق والجودة والحوكمة</h1><p>استقبل أحداث Kafka مع نقطة تحقق مستمرة، وطابق سجلات الوصول وهويات الأحداث، وافحص الدفعة واعزل المعيب وأعد الفحص، ووثّق قرارات الجودة والحوكمة.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [30]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>05 · Receive Kafka events</h2><p>Follow <a href="labs/lab05/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>05 · استقبل أحداث Kafka</h2><p>اتبع <a href="labs/lab05/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [31]:
from pathlib import Path
import os, sys, subprocess, socket, time

ROOT = Path("/content/masar-modern-data-engineering")

if not (ROOT / "course.json").is_file():
    raise RuntimeError(
        "ملفات المشروع غير موجودة. افتح جلسة Colab التي عملت فيها على الأيام السابقة."
    )

os.chdir(ROOT)

print("1/3 تثبيت متطلبات اليوم الرابع...", flush=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(ROOT / "requirements-day04.txt")
    ],
    check=True
)

from kafka.admin import KafkaAdminClient

BASE = Path("/content/masar-kafka-local")
HOME_KAFKA = BASE / "kafka_2.13-4.0.2"
DATA = BASE / "data"
CONFIG = BASE / "server.properties"
LOG = BASE / "server.log"

BASE.mkdir(exist_ok=True)

def port_open():
    try:
        with socket.create_connection(("127.0.0.1", 9092), timeout=1):
            return True
    except OSError:
        return False

print("2/3 تجهيز Kafka داخل جلسة Colab...", flush=True)

if not port_open():
    if not (HOME_KAFKA / "bin/kafka-server-start.sh").is_file():
        archive = BASE / "kafka.tgz"

        subprocess.run(
            [
                "curl", "-fL", "--retry", "2",
                "-o", str(archive),
                "https://archive.apache.org/dist/kafka/4.0.2/kafka_2.13-4.0.2.tgz"
            ],
            check=True
        )

        subprocess.run(
            ["tar", "-xzf", str(archive), "-C", str(BASE)],
            check=True
        )

    CONFIG.write_text(
        f"""process.roles=broker,controller
node.id=1
controller.quorum.bootstrap.servers=127.0.0.1:9093
listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
advertised.listeners=PLAINTEXT://127.0.0.1:9092
controller.listener.names=CONTROLLER
inter.broker.listener.name=PLAINTEXT
listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
log.dirs={DATA}
num.partitions=1
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
group.initial.rebalance.delay.ms=0
""",
        encoding="utf-8"
    )

    env = os.environ.copy()
    env["KAFKA_HEAP_OPTS"] = "-Xms256m -Xmx512m"
    storage = str(HOME_KAFKA / "bin/kafka-storage.sh")

    if not (DATA / "meta.properties").exists():
        if DATA.exists() and any(DATA.iterdir()):
            raise RuntimeError(
                "توجد بيانات Kafka سابقة غير مكتملة. أرسل هذه الرسالة قبل المتابعة."
            )

        cluster = subprocess.check_output(
            [storage, "random-uuid"],
            env=env,
            text=True
        ).strip()

        subprocess.run(
            [
                storage, "format", "--standalone",
                "-t", cluster, "-c", str(CONFIG)
            ],
            env=env,
            check=True
        )

    with LOG.open("a") as output:
        process = subprocess.Popen(
            [
                str(HOME_KAFKA / "bin/kafka-server-start.sh"),
                str(CONFIG)
            ],
            stdout=output,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True
        )

print("3/3 التحقق من الاتصال...", flush=True)

ready = False

for attempt in range(30):
    try:
        client = KafkaAdminClient(
            bootstrap_servers="127.0.0.1:9092",
            api_version_auto_timeout_ms=3000,
            request_timeout_ms=10000
        )
        try:
            client.list_topics()
        finally:
            client.close()

        ready = True
        break

    except Exception:
        time.sleep(2)

if not ready:
    if LOG.exists():
        print(LOG.read_text(errors="replace")[-6000:])

    raise RuntimeError(
        "لم يكتمل تشغيل Kafka. أرسل آخر رسالة ظهرت في هذه الخلية."
    )

subprocess.run(
    [sys.executable, "scripts/run_day04.py", "--preflight"],
    check=True
)

print("✅ اكتمل فحص الإعداد. شغّل الآن خلية التمرين الموجودة أسفل هذه الخلية.")

1/3 تثبيت متطلبات اليوم الرابع...
2/3 تجهيز Kafka داخل جلسة Colab...
3/3 التحقق من الاتصال...


ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disc

✅ اكتمل فحص الإعداد. شغّل الآن خلية التمرين الموجودة أسفل هذه الخلية.


In [32]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = Path("/content/masar-modern-data-engineering")

if "SOURCE" not in globals() or "WORK" not in globals():
    raise RuntimeError(
        "شغّل أول خلية إعداد في دفتر اليوم الرابع، ثم أعد تشغيل هذه الخلية."
    )

# تحميل مكتبات Spark المطلوبة منذ بداية التشغيل
env = os.environ.copy()

for name in ("PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"):
    env.pop(name, None)

env["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.3.3,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 "
    "pyspark-shell"
)

code = r'''
from pathlib import Path
import sys
import json
import pyspark

ROOT = Path(sys.argv[1])
SOURCE = Path(sys.argv[2])
WORK = Path(sys.argv[3])

sys.path.insert(0, str(ROOT / "src"))

if pyspark.__version__ != "3.5.8":
    raise RuntimeError(
        "إصدار Spark مختلف عن إصدار الدورة: " + pyspark.__version__
    )

from masar.runtime import start_spark
from masar.native_contracts import validate_stage_result
from kafka.admin import KafkaAdminClient
import masar.streaming as streaming

print("بدء Spark وتحميل مكتبة Kafka...", flush=True)
spark = start_spark(WORK, kafka=True)

try:
    # فحص وجود موصل Kafka دون تشغيل تدفق أو نشر رسائل
    probe = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "127.0.0.1:9092")
        .option("subscribePattern", ".*")
        .load()
    )

    print("✅ تم تحميل موصل Kafka.", flush=True)

    # حماية رسائل المحاولات السابقة من إعادة الإرسال
    admin = KafkaAdminClient(
        bootstrap_servers="127.0.0.1:9092",
        api_version_auto_timeout_ms=5000,
        request_timeout_ms=10000
    )
    try:
        previous_topics = set(admin.list_topics())
    finally:
        admin.close()

    original_publish = streaming.publish_fixture

    def guarded_publish(source, topic, *args, **kwargs):
        if topic in previous_topics:
            raise RuntimeError(
                "توقّف التمرين لحماية رسائل محاولة سابقة. "
                "أرسل هذه الرسالة دون حذف أي ملفات. Topic: " + topic
            )
        return original_publish(source, topic, *args, **kwargs)

    streaming.publish_fixture = guarded_publish

    print("تشغيل محاولة جديدة للتمرين...", flush=True)

    result = streaming.run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result("lab05_streaming", result)

    print(json.dumps({
        "scope": result["scope"],
        "checks": result["checks"]
    }, indent=2))

    print(
        "Transport rows:",
        [phase["transport_rows"] for phase in result["phases"]]
    )

    print(
        "Unique event IDs:",
        [phase["unique_event_ids"] for phase in result["phases"]]
    )

    (
        spark.read.format("delta")
        .load(str(WORK / result["event_table"]))
        .select("event_id", "trip_id", "event_ts")
        .orderBy("event_id")
        .show(5, truncate=False)
    )

finally:
    spark.stop()
'''

print("جارٍ تحميل المكتبات وتشغيل التمرين؛ انتظر انتهاء الخلية.", flush=True)

process = subprocess.Popen(
    [
        sys.executable, "-u", "-c", code,
        str(ROOT), str(Path(SOURCE).resolve()), str(Path(WORK).resolve())
    ],
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError(
        "لم يكتمل التمرين. أرسل آخر 20 سطرًا ظهرت فوق هذه الرسالة."
    )

print("✅ اكتمل التمرين بنجاح. يمكنك تشغيل الخلية التالية.")

جارٍ تحميل المكتبات وتشغيل التمرين؛ انتظر انتهاء الخلية.
بدء Spark وتحميل مكتبة Kafka...
https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-71d452f2-90ed-4fb9-8a2f-7c6284b0b42d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.8 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.8 in central
	found org.apache.kafka#kafka-clients;3.4.1 i

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>06 · Validate and quarantine</h2><p>Follow <a href="labs/lab06/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>06 · افحص واعزل السجلات</h2><p>اتبع <a href="labs/lab06/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [3]:
from google.colab import files
from pathlib import Path
import hashlib, io, zipfile, subprocess, sys

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("اختر ملف masar_colab_recovery.zip فقط.")

data = next(iter(uploaded.values()))
expected = "ffe0748aebeee1e00c07db6abb9be1022b47c932d27a12915d3c1d02e5df3d58"

if hashlib.sha256(data).hexdigest() != expected:
    raise RuntimeError("الملف مختلف. اختر ملف الاستعادة المرفق في المحادثة.")

folder = Path("/content/masar_recovery")
folder.mkdir(exist_ok=True)

with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(folder)

process = subprocess.Popen(
    [sys.executable, "-u", str(folder / "restore_masar.py")],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("توقف الإعداد. أرسل آخر الرسائل الظاهرة أعلاه.")

Saving masar_colab_recovery.zip to masar_colab_recovery.zip
1/4 تنزيل كود مشروعك من GitHub...
2/4 استعادة مخرجاتك السابقة...
الملفات المستعادة: 504
3/4 تجهيز Python 3.11 وJava 17 ومتطلبات فحص الجودة...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 45.4 MB/s eta 0:00:00
Using CPython 3.11.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: masar-course-py311
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source masar-course-py311/bin/activate
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(R

In [4]:
from pathlib import Path
from google.colab import files
import subprocess

process = subprocess.Popen(
    [
        "/content/masar-course-py311/bin/python",
        "-u",
        "/content/masar_recovery/run_quality.py"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("لم يكتمل الفحص. أرسل آخر الرسائل الظاهرة أعلاه.")

saved_zip = Path(
    "/content/masar_recovery/latest_handoff_path.txt"
).read_text().strip()

files.download(saved_zip)

تشغيل محاولة جديدة لفحص الجودة؛ انتظر انتهاء الخلية.
Python: 3.11.13
https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/content/masar-course-py311/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-77276f27-98c9-44f2-886a-b146db102167;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.3/delta-spark_2.12-3.3.3.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.3.3!delta-spark_2.12.jar (395ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.3/delt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [34]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day04_handoff.zip
